# 02 · Evaluation — Inference Test on Held-Out Set

Load the exported ONNX model and run inference on the test split. Compute threshold sweep, confusion matrix, PR curve, and score histograms. Threshold 0.98 is selected as the operational operating point.

In [18]:
import torch
import torch.nn as nn
import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import models
import glob

In [19]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# Mismo transform que usaste en eval (sin augmentations)
eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_LINEAR),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

In [20]:
class DeepfakeClassifier(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None
        )

        for p in resnet.parameters():
            p.requires_grad = False
        # for p in resnet.layer2.parameters():
        #     p.requires_grad = True
        for p in resnet.layer3.parameters():
            p.requires_grad = True
        for p in resnet.layer4.parameters():
            p.requires_grad = True

        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.avgpool = resnet.avgpool

        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(2048, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.head(x).squeeze(1)

In [21]:
def load_model(ckpt_path):
    model = DeepfakeClassifier(pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    return model


@torch.no_grad()
def predict(model, image_paths, threshold=0.5):
    results = []
    for path in image_paths:
        # Cargar con PIL y convertir a numpy RGB
        img = np.array(Image.open(path).convert('RGB'))
        
        tensor = eval_transform(image=img)['image'].unsqueeze(0).to(DEVICE)
        
        logit = model(tensor)
        score = torch.sigmoid(logit).item()
        pred = int(score >= threshold)
        
        results.append({
            'path': str(path),
            'score': score,
            'pred': pred,
            'label': 'deepfake' if pred == 1 else 'real',
        })
    return results



In [23]:
model = load_model('./05 Checkpoints/best_model_r50.pth')

In [24]:
image_paths = ['thispersodontexist.png']

results = predict(model, image_paths, threshold=0.5)

for r in results:
    print(f"{r['path']:40s}  score={r['score']:.4f}  →  {r['label'].upper()}")

thispersodontexist.png                    score=0.0000  →  REAL


In [25]:
from DeepFakeDetector import DeepfakeDetector

detector = DeepfakeDetector('./09 Final Model/faceswapp_detector.onnx')

img = Image.open('thispersodontexist.png')
result = detector.predict(img)
print(result)

{'score': 1.2718183711041275e-12, 'pred': 0, 'label': 'real', 'threshold': 0.98}
